# T-DEED：補黑幀會不會讓分數變低？（CS 496）

上一次評估中，每段 GAR 片段只有 4.8 秒，T-DEED 卻一次要看 8 秒，所以補了黑色畫面。
這個 notebook 把**同一場比賽裡相鄰的片段接回連續影片**，讓每個動作前後都有真實畫面，
再拿**同一批動作**比較兩種情況：

- **padded**：單獨一段 4.8 秒片段 + 黑幀（跟上次一樣）
- **context**：接好的連續影片

**開始之前：** 「執行階段 → 變更執行階段類型」→ **T4 GPU**。第 5 格貼上 token 之後，可以點第 6 格，用「執行階段 → 執行之後的儲存格」讓它自己跑完。

預設跑 1 個 shard（358 個動作，同一場比賽），大約 **40 分鐘**。

## 1. 確認有 GPU

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv
import torch
assert torch.cuda.is_available(), "沒有 GPU：請到「執行階段 → 變更執行階段類型」選 T4 GPU，然後重新執行這一格"
print("GPU OK:", torch.cuda.get_device_name(0))

## 2. 安裝 T-DEED 和程式碼

In [ ]:
%cd /content
!rm -rf TEED-match-tagger
!git clone -q https://github.com/jmoussa99/TEED-match-tagger.git
!git clone -q https://github.com/arturxe2/T-DEED TEED-match-tagger/private/T-DEED
!git -C TEED-match-tagger/private/T-DEED checkout -q d953b4de8a2ad941c565e1683c5b07255cbe644e
!git -C TEED-match-tagger/private/T-DEED apply /content/TEED-match-tagger/app/patches/tdeed_mps.patch
!pip -q install timm gdown
CKPT_DIR = "/content/TEED-match-tagger/private/T-DEED/checkpoints/SoccerNetBall/SoccerNetBall_challenge2"
!mkdir -p {CKPT_DIR}
!gdown -q 1uCWam8EhAXt3z6QQAfhRfzHITUcml438 -O {CKPT_DIR}/checkpoint_best.pt
!ls -lh {CKPT_DIR}

## 3. 評估腳本（還沒推上 GitHub，所以直接寫進來）

In [ ]:
%%writefile /content/TEED-match-tagger/scripts/evaluate_gar.py
"""Evaluate the T-DEED tagger on labeled SoccerNet-GAR video clips.

Every GAR clip is 4.8 s long and its labeled action happens at ~2.43 s (the clip center).
For each clip we run the same scoring pipeline as the app (app/tdeed_runner.compute_scores),
take each class's highest score within +/- WINDOW_S seconds of the action time, and predict
the class with the highest score. Predictions are restricted to the 10 GAR classes by default
(T-DEED's extra classes DRIVE and BALL PLAYER BLOCK do not exist in GAR).

Usage (from this repo's root, after scripts/setup_tdeed.sh):
  .venv/bin/python scripts/evaluate_gar.py                       # every clip found in --videos
  .venv/bin/python scripts/evaluate_gar.py --per-class 20        # at most 20 clips per class
  .venv/bin/python scripts/evaluate_gar.py --limit 10            # quick smoke test

Outputs (outputs/gar_eval/): predictions.csv, per_class.csv, confusion.csv, metrics.json
"""
from __future__ import annotations

import argparse
import json
import random
import sys
import time
from collections import Counter
from pathlib import Path

import numpy as np
import pandas as pd

REPO_ROOT = Path(__file__).resolve().parents[1]
sys.path.insert(0, str(REPO_ROOT / "app"))
import tdeed_runner as tr  # noqa: E402

if sys.platform != "darwin":
    # The app decodes with macOS VideoToolbox ("-hwaccel videotoolbox"), which does not exist on
    # Linux (e.g. Google Colab). Re-use the app's decoder with that flag removed.
    import subprocess as _sp
    _orig_popen = _sp.Popen

    def _popen_without_videotoolbox(cmd, *args, **kwargs):
        if isinstance(cmd, list) and "videotoolbox" in cmd:
            i = cmd.index("videotoolbox")
            cmd = cmd[: i - 1] + cmd[i + 1:]
        return _orig_popen(cmd, *args, **kwargs)

    tr.subprocess.Popen = _popen_without_videotoolbox

GAR_PROJECT = REPO_ROOT.parent / "sn-gar-2027"
ACTION_TIME_S = 2.43   # alignment.action_offset_seconds in the GAR video metadata
WINDOW_S = 1.0         # +/- 1 s, the same tolerance as the SoccerNet mAP@1 metric
OUT_DIR = REPO_ROOT / "outputs" / "gar_eval"


def load_labels(annotations: Path) -> dict[str, str]:
    data = json.load(open(annotations))["data"]
    return {s["id"]: s["labels"]["action"]["label"] for s in data}


def clip_id(path: Path, split: str) -> str:
    # clip_000123.mp4 -> valid_000123
    return f"{split}_{path.stem.split('_')[-1]}"


def predict_from_scores(scores: np.ndarray, classes: list[str]) -> tuple[str, float, dict[str, float]]:
    """scores: (N, 1 + 12) per-frame probabilities at tr.SAMPLED_FPS (column 0 = background)."""
    center = int(round(ACTION_TIME_S * tr.SAMPLED_FPS))
    half = int(round(WINDOW_S * tr.SAMPLED_FPS))
    lo, hi = max(0, center - half), min(len(scores), center + half + 1)
    window = scores[lo:hi, 1:]                      # drop background
    peak = window.max(axis=0) if len(window) else np.zeros(len(tr.CLASSES))
    per_class = {c: float(peak[i]) for i, c in enumerate(tr.CLASSES)}
    best = max(classes, key=lambda c: per_class[c])
    return best, per_class[best], per_class


def main() -> None:
    ap = argparse.ArgumentParser(description=__doc__, formatter_class=argparse.RawDescriptionHelpFormatter)
    ap.add_argument("--split", default="valid", choices=["train", "valid", "test"])
    ap.add_argument("--annotations", type=Path, default=None,
                    help="GAR annotations JSON (default: ../sn-gar-2027/data/annotations_<split>.json)")
    ap.add_argument("--videos", type=Path, default=None,
                    help="folder of GAR clips (default: ../sn-gar-2027/data/videos/<split>)")
    ap.add_argument("--per-class", type=int, default=None, help="max clips per class")
    ap.add_argument("--limit", type=int, default=None, help="max clips in total")
    ap.add_argument("--all-12", action="store_true", help="allow DRIVE / BALL PLAYER BLOCK predictions")
    ap.add_argument("--device", default="auto", choices=["auto", "cuda", "mps", "cpu"],
                    help="auto = cuda if available, else mps, else cpu")
    ap.add_argument("--seed", type=int, default=0)
    a = ap.parse_args()

    annotations = a.annotations or GAR_PROJECT / "data" / f"annotations_{a.split}.json"
    videos_dir = a.videos or GAR_PROJECT / "data" / "videos" / a.split
    labels = load_labels(annotations)

    clips = sorted(videos_dir.glob("clip_*.mp4"))
    clips = [p for p in clips if clip_id(p, a.split) in labels]
    if not clips:
        sys.exit(f"No labeled clips found in {videos_dir}")
    if a.per_class:
        rng = random.Random(a.seed)
        by_class: dict[str, list[Path]] = {}
        for p in clips:
            by_class.setdefault(labels[clip_id(p, a.split)], []).append(p)
        clips = sorted(p for ps in by_class.values() for p in rng.sample(ps, min(a.per_class, len(ps))))
    if a.limit:
        clips = clips[: a.limit]
    print(f"{len(clips)} clips:", dict(Counter(labels[clip_id(p, a.split)] for p in clips).most_common()))

    classes = tr.CLASSES if a.all_12 else tr.GAR_CLASSES
    device = a.device
    if device == "auto" and tr.torch.cuda.is_available():
        device = "cuda"
    model = tr.get_model(device)
    print(f"Model on {tr.pick_device(device)}")
    rows, t0 = [], time.perf_counter()
    for i, p in enumerate(clips, 1):
        scores = tr.compute_scores(p, model=model)
        pred, conf, per_class = predict_from_scores(scores, classes)
        cid = clip_id(p, a.split)
        rows.append({"id": cid, "label": labels[cid], "pred": pred, "confidence": round(conf, 4),
                     **{f"score_{c}": round(v, 4) for c, v in per_class.items()}})
        if i % 10 == 0 or i == len(clips):
            el = time.perf_counter() - t0
            print(f"  {i}/{len(clips)} clips · {el / i:.1f} s per clip · ETA {el / i * (len(clips) - i) / 60:.1f} min",
                  flush=True)

    df = pd.DataFrame(rows)
    OUT_DIR.mkdir(parents=True, exist_ok=True)
    df.to_csv(OUT_DIR / "predictions.csv", index=False)

    present = [c for c in tr.GAR_CLASSES if (df.label == c).any()]
    recall = {c: float((df[df.label == c].pred == c).mean()) for c in present}
    precision = {c: float((df[df.pred == c].label == c).mean()) if (df.pred == c).any() else float("nan")
                 for c in present}
    per_class = pd.DataFrame({"class": present,
                              "n_clips": [int((df.label == c).sum()) for c in present],
                              "recall": [round(recall[c], 3) for c in present],
                              "precision": [round(precision[c], 3) for c in present]})
    per_class.to_csv(OUT_DIR / "per_class.csv", index=False)
    pd.crosstab(df.label, df.pred).to_csv(OUT_DIR / "confusion.csv")
    metrics = {"split": a.split, "n_clips": len(df), "classes_present": len(present),
               "prediction_classes": "all 12" if a.all_12 else "GAR 10",
               "window_seconds": WINDOW_S, "action_time_seconds": ACTION_TIME_S,
               "balanced_accuracy": 100 * float(np.mean(list(recall.values()))),
               "accuracy": 100 * float((df.label == df.pred).mean()),
               "seconds_per_clip": round((time.perf_counter() - t0) / len(df), 2)}
    json.dump(metrics, open(OUT_DIR / "metrics.json", "w"), indent=2)

    print(json.dumps(metrics, indent=2))
    print(per_class.to_string(index=False))
    print(f"\nSaved to {OUT_DIR}")


if __name__ == "__main__":
    main()

In [ ]:
%%writefile /content/TEED-match-tagger/scripts/evaluate_gar_context.py
"""Does padding hurt T-DEED? Compare the same GAR events with and without real context.

GAR clips are 4.8 s with the action at ~2.43 s, but T-DEED looks at 8 s windows, so a lone clip
is padded with blank frames. Consecutive GAR clips from the same game overlap in the source
match video, so we stitch them back into continuous footage (frame-exact, using each clip's
source frame indices) and score every event twice:

  padded  : the lone 4.8 s clip, exactly as scripts/evaluate_gar.py does
  context : the stitched continuous segment that contains the clip

Same model, same events, same decision rule (best GAR class within +/-1 s of the action).

Input: a folder of GAR clips with their sidecar JSON files (clip_XXXXXX.mp4 + clip_XXXXXX.json,
as stored in the dataset's video shards).

Usage:
  .venv/bin/python scripts/evaluate_gar_context.py --clips DIR
  .venv/bin/python scripts/evaluate_gar_context.py --clips DIR --stitch-only     # just build/check videos
Outputs (outputs/gar_context/): events.csv, metrics.json, per_class.csv, stitched/*.mp4
"""
from __future__ import annotations

import argparse
import json
import subprocess
import sys
import time
from pathlib import Path

import numpy as np
import pandas as pd

sys.path.insert(0, str(Path(__file__).resolve().parent))
import evaluate_gar as eg  # noqa: E402  (also applies the Linux ffmpeg fix)

tr = eg.tr
SOURCE_FPS = 30000 / 1001
OUT_DIR = eg.REPO_ROOT / "outputs" / "gar_context"


def load_clips(clips_dir: Path) -> pd.DataFrame:
    rows = []
    for js in sorted(clips_dir.glob("clip_*.json")):
        mp4 = js.with_suffix(".mp4")
        if not mp4.is_file():
            continue
        j = json.loads(js.read_text())
        c = j["metadata"]["conversion"]
        rows.append({"id": j["id"], "label": j["labels"]["action"]["label"], "game": j["metadata"]["game_id"],
                     "path": str(mp4), "start_frame": c["clip"]["start_frame"],
                     "end_frame": c["clip"]["end_frame_exclusive"],
                     "action_offset_s": c["alignment"]["action_offset_seconds"]})
    if not rows:
        sys.exit(f"No clip_*.mp4 + clip_*.json pairs in {clips_dir}")
    return pd.DataFrame(rows).sort_values(["game", "start_frame"]).reset_index(drop=True)


def find_segments(df: pd.DataFrame) -> list[pd.DataFrame]:
    """Group clips into runs whose source frames overlap or touch (no gaps)."""
    segs, cur, cur_end = [], [], None
    for i, r in df.iterrows():
        if cur and r.game == df.loc[cur[0], "game"] and r.start_frame <= cur_end:
            cur.append(i); cur_end = max(cur_end, r.end_frame)
        else:
            if cur:
                segs.append(df.loc[cur])
            cur, cur_end = [i], r.end_frame
    segs.append(df.loc[cur])
    return segs


def stitch(seg: pd.DataFrame, out: Path) -> tuple[int, int]:
    """Concatenate the segment's clips frame-exactly. Returns (first source frame, n frames)."""
    seg = seg.sort_values("start_frame")
    starts, ends = seg.start_frame.tolist(), seg.end_frame.tolist()
    first, last = starts[0], max(ends)
    parts, pos = [], first                      # pos = next source frame still missing
    for path, s, e in zip(seg.path, starts, ends):
        if e <= pos:
            continue                            # fully covered already
        parts.append((path, pos - s, e - s))    # take [pos, e) from this clip
        pos = e
    if out.is_file():
        return first, last - first
    cmd = ["ffmpeg", "-v", "error", "-y", "-nostdin"]
    for path, _, _ in parts:
        cmd += ["-i", path]
    chains = [f"[{k}:v]trim=start_frame={a}:end_frame={b},setpts=PTS-STARTPTS[v{k}]"
              for k, (_, a, b) in enumerate(parts)]
    concat = "".join(f"[v{k}]" for k in range(len(parts))) + f"concat=n={len(parts)}:v=1:a=0[out]"
    cmd += ["-filter_complex", ";".join(chains + [concat]), "-map", "[out]",
            "-vsync", "passthrough", "-c:v", "libx264", "-preset", "veryfast", "-crf", "16", "-pix_fmt", "yuv420p", str(out)]
    subprocess.run(cmd, check=True)
    return first, last - first


def count_frames(video: Path) -> int:
    out = subprocess.run(["ffprobe", "-v", "error", "-count_frames", "-select_streams", "v:0",
                          "-show_entries", "stream=nb_read_frames", "-of", "csv=p=0", str(video)],
                         capture_output=True, text=True, check=True).stdout.strip()
    return int(out)


def predict_at(scores: np.ndarray, center_s: float, classes: list[str]) -> tuple[str, float]:
    center = int(round(center_s * tr.SAMPLED_FPS))
    half = int(round(eg.WINDOW_S * tr.SAMPLED_FPS))
    win = scores[max(0, center - half): min(len(scores), center + half + 1), 1:]
    peak = win.max(axis=0) if len(win) else np.zeros(len(tr.CLASSES))
    best = max(classes, key=lambda c: peak[tr.CLASSES.index(c)])
    return best, float(peak[tr.CLASSES.index(best)])


def balanced_acc(df: pd.DataFrame, col: str) -> float:
    classes = [c for c in tr.GAR_CLASSES if (df.label == c).any()]
    return 100 * float(np.mean([(df[df.label == c][col] == c).mean() for c in classes])) if classes else float("nan")


def main() -> None:
    ap = argparse.ArgumentParser(description=__doc__, formatter_class=argparse.RawDescriptionHelpFormatter)
    ap.add_argument("--clips", type=Path, required=True, help="folder with clip_*.mp4 + clip_*.json")
    ap.add_argument("--device", default="auto", choices=["auto", "cuda", "mps", "cpu"])
    ap.add_argument("--stitch-only", action="store_true", help="build and check stitched videos, no model")
    a = ap.parse_args()

    df = load_clips(a.clips)
    segs = find_segments(df)
    (OUT_DIR / "stitched").mkdir(parents=True, exist_ok=True)
    print(f"{len(df)} clips from {df.game.nunique()} game(s) -> {len(segs)} continuous segments")

    seg_info = []
    for k, seg in enumerate(segs):
        video = OUT_DIR / "stitched" / f"seg_{seg.game.iloc[0]}_{k:04d}.mp4"
        first, n = stitch(seg, video)
        seg_info.append((seg, video, first, n))
    if a.stitch_only:
        bad = [(v.name, n, count_frames(v)) for _, v, _, n in seg_info if count_frames(v) != n]
        print("frame-count check:", "all segments OK" if not bad else f"MISMATCH {bad[:5]}")
        return

    device = "cuda" if a.device == "auto" and tr.torch.cuda.is_available() else a.device
    model = tr.get_model(device)
    print(f"Model on {tr.pick_device(device)}")
    classes = tr.GAR_CLASSES
    rows, t0 = [], time.perf_counter()
    for k, (seg, video, first, n) in enumerate(seg_info, 1):
        seg_scores = tr.compute_scores(video, model=model)
        seg_len_s = n / SOURCE_FPS
        for r in seg.itertuples():
            event_s = (r.start_frame - first) / SOURCE_FPS + r.action_offset_s
            ctx_pred, ctx_conf = predict_at(seg_scores, event_s, classes)
            pad_pred, pad_conf = predict_at(tr.compute_scores(Path(r.path), model=model), r.action_offset_s, classes)
            rows.append({"id": r.id, "label": r.label, "segment": video.name,
                         "context_before_s": round(event_s, 2), "context_after_s": round(seg_len_s - event_s, 2),
                         "pred_padded": pad_pred, "conf_padded": round(pad_conf, 4),
                         "pred_context": ctx_pred, "conf_context": round(ctx_conf, 4)})
        el = time.perf_counter() - t0
        print(f"  segment {k}/{len(seg_info)} · {len(rows)} events · {el / 60:.1f} min elapsed", flush=True)

    ev = pd.DataFrame(rows)
    ev["min_context_s"] = ev[["context_before_s", "context_after_s"]].min(axis=1)
    ev.to_csv(OUT_DIR / "events.csv", index=False)
    full = ev[ev.min_context_s >= 4.0]          # T-DEED window = 8 s -> 4 s on each side
    metrics = {
        "n_events": len(ev), "n_segments": len(seg_info),
        "balanced_accuracy_padded": balanced_acc(ev, "pred_padded"),
        "balanced_accuracy_context": balanced_acc(ev, "pred_context"),
        "accuracy_padded": 100 * float((ev.label == ev.pred_padded).mean()),
        "accuracy_context": 100 * float((ev.label == ev.pred_context).mean()),
        "n_events_full_context_4s": len(full),
        "balanced_accuracy_padded_full_context_subset": balanced_acc(full, "pred_padded"),
        "balanced_accuracy_context_full_context_subset": balanced_acc(full, "pred_context"),
        "fixed_by_context": int(((ev.pred_padded != ev.label) & (ev.pred_context == ev.label)).sum()),
        "broken_by_context": int(((ev.pred_padded == ev.label) & (ev.pred_context != ev.label)).sum()),
    }
    json.dump(metrics, open(OUT_DIR / "metrics.json", "w"), indent=2)
    present = [c for c in tr.GAR_CLASSES if (ev.label == c).any()]
    per = pd.DataFrame({"class": present, "n_events": [int((ev.label == c).sum()) for c in present],
                        "recall_padded": [round((ev[ev.label == c].pred_padded == c).mean(), 3) for c in present],
                        "recall_context": [round((ev[ev.label == c].pred_context == c).mean(), 3) for c in present]})
    per.to_csv(OUT_DIR / "per_class.csv", index=False)
    print(json.dumps(metrics, indent=2))
    print(per.to_string(index=False))
    print(f"\nSaved to {OUT_DIR}")


if __name__ == "__main__":
    main()

## 4. 登入 Hugging Face（貼上 `hf_` 開頭的 Read token）

In [ ]:
from getpass import getpass
from huggingface_hub import login
login(token=getpass("Hugging Face token: "))

## 5. 下載影片片段（含每段的 JSON，用來把片段接回去）

`N_SHARDS = 1` 是 358 個動作；改成 2 會變 730 個左右，時間大約加倍。

In [ ]:
import tarfile, json, os
from pathlib import Path
from huggingface_hub import hf_hub_download

N_SHARDS = 1
REPO = "OpenSportsLab/SoccerNet-GAR"
CLIPS = Path("/content/gar_context/clips"); CLIPS.mkdir(parents=True, exist_ok=True)
for k in range(N_SHARDS):
    shard = f"valid/shards/shard-{k:06d}.tar"
    tar_path = hf_hub_download(REPO, shard, repo_type="dataset", revision="videos", local_dir="/content/hf_tmp")
    pending = {}
    with tarfile.open(tar_path) as tf:
        for mem in tf:
            key, ext = mem.name.split(".", 1)
            data = tf.extractfile(mem).read()
            if ext == "json":
                name = Path(json.loads(data)["inputs"][0]["path"]).stem       # clip_000123
                (CLIPS / f"{name}.json").write_bytes(data)
                if key in pending:
                    (CLIPS / f"{name}.mp4").write_bytes(pending.pop(key))
                else:
                    pending[key] = name
            else:
                if isinstance(pending.get(key), str):
                    (CLIPS / f"{pending.pop(key)}.mp4").write_bytes(data)
                else:
                    pending[key] = data
    os.remove(tar_path)
    print(f"{k + 1}/{N_SHARDS} {shard} done", flush=True)
print(len(list(CLIPS.glob("*.mp4"))), "clips,", len(list(CLIPS.glob("*.json"))), "json files")

## 6. 把片段接回連續影片，並檢查幀數

In [ ]:
%cd /content/TEED-match-tagger
!python scripts/evaluate_gar_context.py --clips /content/gar_context/clips --stitch-only

## 7. 評估：同一批動作，padded vs context（約 35 分鐘）

In [ ]:
%cd /content/TEED-match-tagger
!python scripts/evaluate_gar_context.py --clips /content/gar_context/clips

## 8. 結果

In [ ]:
import json, pandas as pd
R = "/content/TEED-match-tagger/outputs/gar_context"
m = json.load(open(f"{R}/metrics.json")); ev = pd.read_csv(f"{R}/events.csv"); pc = pd.read_csv(f"{R}/per_class.csv")
print(json.dumps(m, indent=2))
display(pc)
full = ev[ev.min_context_s >= 4]
print(f"\nEvents with >= 4 s of real footage on both sides: {len(full)}")
print("accuracy padded %.1f%% -> context %.1f%%" % (100 * (full.label == full.pred_padded).mean(), 100 * (full.label == full.pred_context).mean()))

## 9. 下載結果（不含接好的影片，檔案很小）

In [ ]:
!cd /content/TEED-match-tagger/outputs && zip -qr /content/gar_context_results.zip gar_context -x "gar_context/stitched/*"
from google.colab import files
files.download("/content/gar_context_results.zip")